In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install tensorflow-addons

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 4.0 MB/s eta 0:00:00


In [ ]:
SEED = 61

import os
import re
import gc
import h5py
import torch
import numpy as np
import pandas as pd
import tensorflow as tf
import random as python_random
import matplotlib.pyplot as plt
import tensorflow_addons as tfa

from tqdm import tqdm
from nltk import tokenize

from sklearn import preprocessing
from sklearn.decomposition import PCA
from IPython.display import display_html
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import BorderlineSMOTE

from keras import backend as K
from keras import initializers,regularizers,constraints
from keras.preprocessing.text import Tokenizer, text_to_word_sequence
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.layers import Reshape, Input, Embedding, Flatten, Dense, Dropout, BatchNormalization, Activation, RepeatVector, Permute
from keras.layers import TimeDistributed, LSTM, GRU, Bidirectional, Convolution1D, MaxPooling1D, MaxPool2D, Convolution2D
from keras.layers import RepeatVector, Reshape
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from keras.models import Sequential, Model, load_model
from sklearn.model_selection import cross_val_score

def reset_seeds():
    np.random.seed(SEED)
    python_random.seed(SEED)
    tf.random.set_seed(SEED)
    os.environ["PYTHONHASHSEED"] = str(SEED)

# from tensorflow.python.keras.layers import Layer, InputSpec, Lambda
# from tensorflow.keras import Model
# from attention import Attention_input1, Attention_input2
# from keras.optimizers import SGD, RMSprop, Adagrad

/usr/local/lib/python3.10/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [ ]:
def read_hdf5(path):
    read_file = h5py.File(path, 'r')

    feature_names = list(read_file.keys())
    loaded_data = []

    for name in feature_names:
        dataset = read_file[name][:]
        if dataset.dtype == np.dtype('object'):
            dataset = np.array([x.decode('UTF-8') for x in dataset])
        loaded_data.append((name, dataset))

    return loaded_data

def loadz(path):
    data = np.load(path)['arr_0']
    return data

In [ ]:
def load_labels(path):
    data = read_hdf5(path)

    for x in data:
        if x[0] == 'multimodal-labels':
            labels = x[1]
        if x[0] == 'text-labels':
            text_labels = x[1]
        if x[0] == 'image-labels':
            image_labels = x[1]

    return labels, text_labels, image_labels

def load_mvsa_feature(feature_name, merge=False):
    folder_path = os.path.join('/content/drive/MyDrive/MVSA_features/', feature_name)
    single_file = 'mvsa-single-{}.npz'.format(feature_name)
    multiple_file = 'mvsa-multiple-{}.npz'.format(feature_name)
    mvsa_single = loadz(os.path.join(folder_path, single_file))
    mvsa_multiple = loadz(os.path.join(folder_path, multiple_file))

    if merge == True:
        return merge_mvsa(mvsa_single, mvsa_multiple)

    return mvsa_single, mvsa_multiple

def load_mvsa_images(merge=False):
    folder_path = '/content/drive/MyDrive/mvsa-data'
    file_paths = os.listdir(folder_path)
    for path in file_paths:
        file_name = os.path.split(path)[1]
        if file_name.split('.')[1] == 'npz':
            if file_name.split('-')[1] == 'single':
                mvsa_single_images_path = os.path.join(folder_path, path)
            else:
                mvsa_multiple_images_path = os.path.join(folder_path, path)

    mvsa_single = loadz(mvsa_single_images_path)
    mvsa_multiple = loadz(mvsa_multiple_images_path)

    if merge == True:
        return merge_mvsa(mvsa_single, mvsa_multiple)

    return mvsa_single, mvsa_multiple

def merge_mvsa(mvsa_single, mvsa_multiple):
    mvsa = np.concatenate((mvsa_single, mvsa_multiple), axis=0)
    return mvsa

In [ ]:
def plot_metrics(history):
    fig = plt.figure(figsize=(20, 5))

    fig.add_subplot(1, 4, 1)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('LOSS')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'val'], loc='best')

    fig.add_subplot(1, 4, 2)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('ACCURACY')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'val'], loc='best')

    fig.add_subplot(1, 4, 3)
    plt.plot(history.history['f1_macro'])
    plt.plot(history.history['val_f1_macro'])
    plt.title('Macro F1-SCORE')
    plt.ylabel('f1-macro')
    plt.xlabel('epoch')
    plt.legend(['train', 'val'], loc='best')

    fig.add_subplot(1, 4, 4)
    plt.plot(history.history['f1_weighted'])
    plt.plot(history.history['val_f1_weighted'])
    plt.title('Weighted F1-SCORE')
    plt.ylabel('f1-weighted')
    plt.xlabel('epoch')
    plt.legend(['train', 'val'], loc='best')

    plt.show()

In [ ]:
# e.g. validation_split=0.1 -----> 8:1:1 ratio of train, val, test
def split_data(data, validation_split):
    num_val = int(validation_split * len(data))
    data_train = data[:-(num_val*2)]
    data_val = data[-(num_val*2):-(num_val)]
    data_test = data[-num_val:]
    return data_train, data_val, data_test

# e.g. validation_split=0.1 -----> 8:1:1 ratio of train, val, test
def split_tf_data(data, validation_split):
    DATASET_SIZE = len(data)
    train_size = int((1-validation_split*2) * DATASET_SIZE)
    val_size = int(validation_split * DATASET_SIZE)
    test_size = int(validation_split * DATASET_SIZE)

#     full_dataset = tf.data.TFRecordDataset(FLAGS.input_file)
#     full_dataset = full_dataset.shuffle()
    train_dataset = data.take(train_size)
    test_dataset = data.skip(train_size)
    val_dataset = test_dataset.skip(test_size)
    test_dataset = test_dataset.take(test_size)
    return train_dataset, val_dataset, test_dataset

In [ ]:
NUM_CLASSES = 3
f1_macro = tfa.metrics.F1Score(num_classes=NUM_CLASSES, average='macro', name='f1_macro')
f1_weighted = tfa.metrics.F1Score(num_classes=NUM_CLASSES, average='weighted', name='f1_weighted')

#     # soft attention
#     attention = Dense(1, activation='tanh') (image_input)
#     attention = Flatten() (attention)
#     attention = Activation('softmax') (attention)
#     attention = RepeatVector(NUM_HIDDEN) (attention)
#     attention = Permute([2, 1]) (attention)
#     attention = Flatten() (attention)

def create_model_pretrained(input_shape, lstm=True):
    image_input = Input(shape=input_shape)
    dropout = Dropout(DROPOUT_INPUT_IMG) (image_input)
    if lstm == True:
        image_reshape = Reshape((1, -1)) (dropout)
        image_lstm = LSTM(NUM_LSTM_IMG) (image_reshape)
        dropout = Dropout(DROPOUT_LSTM_IMG) (image_lstm)
    outputs = Dense(NUM_CLASSES, activation='softmax') (dropout)
    model = Model(image_input, outputs)
    model.compile(optimizer=OPTIMIZER, loss=LOSS, metrics=['accuracy', f1_macro, f1_weighted])
    return model

In [ ]:
def evaluate_model(model, X_test, y_test, checkpoint=None, verbose=1):
    if checkpoint is not None:
        model = load_model('./model_checkpoint/{}.h5'.format(checkpoint))

    loss, acc, f1_macro, f1_weighted = model.evaluate(X_test, y_test, verbose=verbose)

    if verbose == 1:
        print('Loss:', loss)
        print('Accuracy:', acc)
        print('Macro F1-score:', f1_macro)
        print('Weighted F1-score:', f1_weighted)

    return loss, acc, f1_macro, f1_weighted

In [ ]:
def run_and_evaluate(name, X, y, verbose=0, lstm=False):
    y = le.fit_transform(y)
    y = to_categorical(np.asarray(y))

    X_train, X_val, X_test = split_data(X, VALIDATION_SPLIT)
    y_train, y_val, y_test = split_data(y, VALIDATION_SPLIT)

    if 'multiple' in name:
        batch_size = BATCH_SIZE_MULTIPLE
        if DO_SMOTE_MULTIPLE == True:
#             oversample = BorderlineSMOTE(sampling_strategy='minority', random_state=SEED, kind='borderline-2')
            oversample = SMOTE(sampling_strategy='minority', random_state=SEED)
            X_train, y_train = oversample.fit_resample(X_train, y_train)
    else:
        batch_size = BATCH_SIZE_SINGLE # 128
        if DO_SMOTE_SINGLE == True:
#             oversample = BorderlineSMOTE(sampling_strategy='minority', random_state=SEED, kind='borderline-2')
            oversample = SMOTE(sampling_strategy='minority', random_state=SEED)
            X_train, y_train = oversample.fit_resample(X_train, y_train)

    model = create_model_pretrained(X_train.shape[1:], lstm=lstm)
    checkpoint = ModelCheckpoint('./model_checkpoint/{}.h5'.format(name), save_best_only=True, verbose=verbose)
#     early_stopping = EarlyStopping(monitor='val_loss', min_delta=1e-4, patience=PATIENCE)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=PATIENCE)
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                                   epochs=EPOCHS, batch_size=batch_size, verbose=verbose,
                                   callbacks=[checkpoint, reduce_lr])
    best_epoch = np.argmin(history.history['val_loss'])
    print('Checkpoint loaded at epoch:', best_epoch)

    return history, evaluate_model(model, X_test, y_test, checkpoint=name, verbose=verbose)

In [ ]:
def run_and_evaluate_new(name, X, y, verbose=0, lstm=False):
    y = le.fit_transform(y)
    y = to_categorical(np.asarray(y))

    X_train, X_val, X_test = split_data(X, VALIDATION_SPLIT)
    y_train, y_val, y_test = split_data(y, VALIDATION_SPLIT)

    if 'multiple' in name:
        batch_size = BATCH_SIZE_MULTIPLE
        if DO_SMOTE_MULTIPLE == True:
#             oversample = BorderlineSMOTE(sampling_strategy='minority', random_state=SEED, kind='borderline-2')
            oversample = SMOTE(sampling_strategy='minority', random_state=SEED)
            X_train, y_train = oversample.fit_resample(X_train, y_train)
    else:
        batch_size = BATCH_SIZE_SINGLE # 128
        if DO_SMOTE_SINGLE == True:
#             oversample = BorderlineSMOTE(sampling_strategy='minority', random_state=SEED, kind='borderline-2')
            oversample = SMOTE(sampling_strategy='minority', random_state=SEED)
            X_train, y_train = oversample.fit_resample(X_train, y_train)

    model = create_model_pretrained(X_train.shape[1:], lstm=lstm)
    checkpoint = ModelCheckpoint('./model_checkpoint/{}.h5'.format(name), save_best_only=True, verbose=verbose)
#     early_stopping = EarlyStopping(monitor='val_loss', min_delta=1e-4, patience=PATIENCE)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=PATIENCE)
    histories = []
    losses = []
    accs = []
    f1_macros = []
    f1_weighteds = []
    for i in range(10):
        print('Fold:', i)
        X_train_i, X_val_i, X_test_i = split_data(X_train, VALIDATION_SPLIT)
        y_train_i, y_val_i, y_test_i = split_data(y_train, VALIDATION_SPLIT)
        history = model.fit(X_train_i, y_train_i, validation_data=(X_val_i, y_val_i),
                                       epochs=EPOCHS, batch_size=batch_size, verbose=verbose,
                                       callbacks=[checkpoint, reduce_lr])
        best_epoch = np.argmin(history.history['val_loss'])
        print('Checkpoint loaded at epoch:', best_epoch)

        loss, acc, f1_macro, f1_weighted = evaluate_model(model, X_test_i, y_test_i, checkpoint=name, verbose=verbose)
        histories.append(history)
        losses.append(loss)
        accs.append(acc)
        f1_macros.append(f1_macro)
        f1_weighteds.append(f1_weighted)
    return histories, losses, accs, f1_macros, f1_weighteds


In [ ]:
def style_dataframe(dataframe):
    return dataframe.style.highlight_max(subset=['Accuracy', 'F1-macro', 'F1-weighted'], props='color:lawngreen', axis=0)\
                          .highlight_min(subset=['Accuracy', 'F1-macro', 'F1-weighted'], props='color:tomato', axis=0)\
                          .highlight_min(subset=['Loss'], props='color:lawngreen', axis=0)\
                          .highlight_max(subset=['Loss'], props='color:tomato', axis=0)

def style_dataframe_out(dataframe):
    return dataframe.style.highlight_max(subset=['Accuracy', 'F1-weighted'], props='color:lawngreen', axis=0)\
                          .highlight_min(subset=['Accuracy', 'F1-weighted'], props='color:tomato', axis=0)

def display_dataframes(dfs, names=[], index=False):
    def to_df(x):
        if isinstance(x, pd.Series):
            return pd.DataFrame(x)
        else:
            return x
    html_str = ''
    if names:
        html_str += ('<tr>' +
                     ''.join(f'<td style="text-align:center">{name}</td>' for name in names) +
                     '</tr>')
    html_str += ('<tr>' +
                 ''.join(f'<td style="vertical-align:top"> {to_df(df).to_html()}</td>'
                         for df in dfs) +
                 '</tr>')
    html_str = f'<table>{html_str}</table>'
    html_str = html_str.replace('table','table style="display:inline"')
    display_html(html_str, raw=True)
    return html_str

# Load data

In [ ]:
mvsa_single_multimodal_labels, mvsa_single_text_labels, mvsa_single_image_labels = load_labels('/content/drive/MyDrive/MVSA_features/labels/mvsa-single-labels.hdf5')
mvsa_multiple_multimodal_labels, mvsa_multiple_text_labels, mvsa_multiple_image_labels = load_labels('/content/drive/MyDrive/MVSA_features/labels/mvsa-multiple-labels.hdf5')

mvsa_multimodal_labels = merge_mvsa(mvsa_single_multimodal_labels, mvsa_multiple_multimodal_labels)
mvsa_text_labels = merge_mvsa(mvsa_single_text_labels, mvsa_multiple_text_labels)
mvsa_image_labels = merge_mvsa(mvsa_single_image_labels, mvsa_multiple_image_labels)

le = preprocessing.LabelEncoder()
le.fit(mvsa_multimodal_labels)
NUM_CLASSES = len(le.classes_) # =3

In [ ]:
# prepare all features data
#feature_names = ['xception', 'vgg16', 'vgg19', 'resnet50', 'resnet101', 'resnet152', 'densenet121', 'densenet169', 'densenet201']
feature_names = [ 'resnet50', 'resnet101','densenet169', 'densenet201']

mvsa_single_features = []
mvsa_multiple_features = []
mvsa_features = []

for name in tqdm(feature_names):
    data = load_mvsa_feature(name)
    merge_data = merge_mvsa(data[0], data[1])

    mvsa_single_features.append(data[0])
    mvsa_multiple_features.append(data[1])
    mvsa_features.append(merge_data)

100%|██████████| 4/4 [00:12<00:00,  3.21s/it]


In [ ]:
def shuffle_mvsa(mvsa_features, labels, indices):
    shuffled_features = []
#     random_idx = np.random.permutation(len(labels))
    for i in range(len(mvsa_features)):
        x = mvsa_features[i][indices]
        shuffled_features.append(x)
    return shuffled_features, labels[indices]

In [ ]:
# Fix random indices for consistency between other experiments
# mvsa_single_features, mvsa_single_multimodal_labels = shuffle_mvsa(mvsa_single_features, mvsa_single_multimodal_labels, np.load('../input/mvsa-shuffle-indices/mvsa-single-shuffle-indices.npy'))
# mvsa_multiple_features, mvsa_multiple_multimodal_labels = shuffle_mvsa(mvsa_multiple_features, mvsa_multiple_multimodal_labels, np.load('../input/mvsa-shuffle-indices/mvsa-multiple-shuffle-indices.npy'))

# Run models and Evalution display

In [ ]:
reset_seeds()
EPOCHS = 100
VALIDATION_SPLIT = 0.1
PATIENCE = 10

BATCH_SIZE_SINGLE = 128
BATCH_SIZE_MULTIPLE = 256

DO_SMOTE_SINGLE = True
DO_SMOTE_MULTIPLE = True

HAS_LSTM = False
# NUM_LSTM_IMG = 64
DROPOUT_INPUT_IMG = 0.5
DROPOUT_LSTM_IMG = 0.5

OPTIMIZER = 'adam'
LOSS = 'categorical_crossentropy'

## With original image labels

In [ ]:
# print('MVSA-Single: With original image labels')
# mvsa_single_histories = []
# mvsa_single_scores = []
# for i in range(len(feature_names)):
#     print('MVSA-Single:', feature_names[i])
#     if feature_names[i] == 'cnn':
#         history, score = run_and_evaluate_cnn('single-OL-' + feature_names[i], mvsa_single_features[i], mvsa_single_image_labels, verbose=0)
#     else:
#         history, score = run_and_evaluate('single-OL-' + feature_names[i], mvsa_single_features[i], mvsa_single_image_labels, verbose=0)
#     mvsa_single_histories.append(history)
#     mvsa_single_scores.append(score)
#     print()
# df_single_scores = pd.DataFrame(mvsa_single_scores, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)

# print('MVSA-Multiple: With original image labels')
# mvsa_multiple_histories = []
# mvsa_multiple_scores = []
# for i in range(len(feature_names)):
# #     print('MVSA-Multiple:', feature_names[i])
#     if feature_names[i] == 'cnn':
#         history, score = run_and_evaluate_cnn('multiple-OL-' + feature_names[i], mvsa_multiple_features[i], mvsa_multiple_image_labels, verbose=1)
#     else:
#         history, score = run_and_evaluate('multiple-OL-' + feature_names[i], mvsa_multiple_features[i], mvsa_multiple_image_labels, verbose=0)
#     mvsa_multiple_histories.append(history)
#     mvsa_multiple_scores.append(score)
#     print()
# df_multiple_scores = pd.DataFrame(mvsa_multiple_scores, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)

# mvsa_average_scores = np.mean([mvsa_single_scores, mvsa_multiple_scores], axis=0)
# df_average_scores = pd.DataFrame(mvsa_average_scores, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)

## With multimodal labels

In [ ]:
print('MVSA-Single: With multimodal labels')
mvsa_single_histories2 = []
mvsa_single_scores2 = []
for i in range(len(feature_names)):
    print('MVSA-Single:', feature_names[i])
    history, score = run_and_evaluate('single-ML-' + feature_names[i], mvsa_single_features[i], mvsa_single_multimodal_labels,
                                      verbose=0, lstm=HAS_LSTM)
    mvsa_single_histories2.append(history)
    mvsa_single_scores2.append(score)
    print()
df_single_scores2 = pd.DataFrame(mvsa_single_scores2, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)

print('MVSA-Multiple: With multimodal labels')
mvsa_multiple_histories2 = []
mvsa_multiple_scores2 = []
for i in range(len(feature_names)):
    print('MVSA-Multiple:', feature_names[i])
    history, score = run_and_evaluate('multiple-ML-' + feature_names[i], mvsa_multiple_features[i], mvsa_multiple_multimodal_labels,
                                      verbose=0, lstm=HAS_LSTM)
    mvsa_multiple_histories2.append(history)
    mvsa_multiple_scores2.append(score)
    print()
df_multiple_scores2 = pd.DataFrame(mvsa_multiple_scores2, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)

mvsa_average_scores2 = np.mean([mvsa_single_scores2, mvsa_multiple_scores2], axis=0)
df_average_scores2 = pd.DataFrame(mvsa_average_scores2, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)

MVSA-Single: With multimodal labels
MVSA-Single: resnet50


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 12

MVSA-Single: resnet101


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 11

MVSA-Single: densenet169


...

Checkpoint loaded at epoch: 22

MVSA-Multiple: densenet169


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 0

MVSA-Multiple: densenet201


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 7



In [ ]:
accs

[0.904315173625946,
 0.904315173625946,
 0.904315173625946,
 0.904315173625946,
 0.904315173625946,
 0.904315173625946,
 0.904315173625946,
 0.904315173625946,
 0.904315173625946,
 0.904315173625946]

In [ ]:
print('MVSA-Single: With multimodal labels')
mvsa_single_histories2 = []
mvsa_single_scores2 = []
for i in range(len(feature_names)):
    print('MVSA-Single:', feature_names[i])
    history, losses, accs, f1_macros, f1_weighteds = run_and_evaluate_new('single-ML-' + feature_names[i], mvsa_single_features[i], mvsa_single_multimodal_labels,
                                      verbose=0, lstm=HAS_LSTM)
    mvsa_single_histories2.append(history)
    mvsa_single_scores2.append([np.mean(losses),np.mean(accs),np.mean(f1_macros),np.mean(f1_weighteds)])
    print()
df_single_scores2 = pd.DataFrame(mvsa_single_scores2, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)

MVSA-Single: With multimodal labels
MVSA-Single: resnet50
Fold: 0


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 2
Fold: 1
Checkpoint loaded at epoch: 0
Fold: 2
Checkpoint loaded at epoch: 0
Fold: 3
Checkpoint loaded at epoch: 0
Fold: 4
Checkpoint loaded at epoch: 0
Fold: 5
Checkpoint loaded at epoch: 0
Fold: 6
Checkpoint loaded at epoch: 0
Fold: 7
Checkpoint loaded at epoch: 0
Fold: 8
Checkpoint loaded at epoch: 0
Fold: 9
Checkpoint loaded at epoch: 0

MVSA-Single: resnet101
Fold: 0


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 2
Fold: 1
Checkpoint loaded at epoch: 0
Fold: 2
Checkpoint loaded at epoch: 0
Fold: 3
Checkpoint loaded at epoch: 0
Fold: 4
Checkpoint loaded at epoch: 0
Fold: 5
Checkpoint loaded at epoch: 0
Fold: 6
Checkpoint loaded at epoch: 0
Fold: 7
Checkpoint loaded at epoch: 0
Fold: 8
Checkpoint loaded at epoch: 0
Fold: 9
Checkpoint loaded at epoch: 0

MVSA-Single: densenet169
Fold: 0


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 31
Fold: 1
Checkpoint loaded at epoch: 8
Fold: 2
Checkpoint loaded at epoch: 0
Fold: 3
Checkpoint loaded at epoch: 0
Fold: 4
Checkpoint loaded at epoch: 0
Fold: 5
Checkpoint loaded at epoch: 0
Fold: 6
Checkpoint loaded at epoch: 0
Fold: 7
Checkpoint loaded at epoch: 0
Fold: 8
Checkpoint loaded at epoch: 0
Fold: 9
Checkpoint loaded at epoch: 0

MVSA-Single: densenet201
Fold: 0


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 22
Fold: 1
Checkpoint loaded at epoch: 0
Fold: 2
Checkpoint loaded at epoch: 0
Fold: 3
Checkpoint loaded at epoch: 0
Fold: 4
Checkpoint loaded at epoch: 0
Fold: 5
Checkpoint loaded at epoch: 0
Fold: 6
Checkpoint loaded at epoch: 0
Fold: 7
Checkpoint loaded at epoch: 0
Fold: 8
Checkpoint loaded at epoch: 0
Fold: 9
Checkpoint loaded at epoch: 0



In [ ]:
df_single_scores2

,Loss,Accuracy,F1-macro,F1-weighted
resnet50,1.141688,0.056285,0.035524,0.106572
resnet101,1.156888,0.041276,0.026426,0.079279
densenet169,0.410523,0.885553,0.313101,0.939303
densenet201,0.400889,0.904315,0.316585,0.949754


In [ ]:
print('MVSA-Multiple: With multimodal labels')
mvsa_multiple_histories2 = []
mvsa_multiple_scores2 = []
for i in range(len(feature_names)):
    print('MVSA-Multiple:', feature_names[i])
    history, losses, accs, f1_macros, f1_weighteds= run_and_evaluate_new('multiple-ML-' + feature_names[i], mvsa_multiple_features[i], mvsa_multiple_multimodal_labels,
                                      verbose=0, lstm=HAS_LSTM)
    mvsa_multiple_histories2.append(history)
    mvsa_multiple_scores2.append([np.mean(losses),np.mean(accs),np.mean(f1_macros),np.mean(f1_weighteds)])
    print()
df_multiple_scores2 = pd.DataFrame(mvsa_multiple_scores2, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)
df_multiple_scores2
# mvsa_average_scores2 = np.mean([mvsa_single_scores2, mvsa_multiple_scores2], axis=0)
# df_average_scores2 = pd.DataFrame(mvsa_average_scores2, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)

MVSA-Multiple: With multimodal labels
MVSA-Multiple: resnet50
Fold: 0


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 18
Fold: 1
Checkpoint loaded at epoch: 0
Fold: 2
Checkpoint loaded at epoch: 0
Fold: 3
Checkpoint loaded at epoch: 0
Fold: 4
Checkpoint loaded at epoch: 0
Fold: 5
Checkpoint loaded at epoch: 0
Fold: 6
Checkpoint loaded at epoch: 0
Fold: 7
Checkpoint loaded at epoch: 0
Fold: 8
Checkpoint loaded at epoch: 0
Fold: 9
Checkpoint loaded at epoch: 0

MVSA-Multiple: resnet101
Fold: 0


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 4
Fold: 1
Checkpoint loaded at epoch: 0
Fold: 2
Checkpoint loaded at epoch: 0
Fold: 3
Checkpoint loaded at epoch: 0
Fold: 4
Checkpoint loaded at epoch: 0
Fold: 5
Checkpoint loaded at epoch: 0
Fold: 6
Checkpoint loaded at epoch: 0
Fold: 7
Checkpoint loaded at epoch: 0
Fold: 8
Checkpoint loaded at epoch: 0
Fold: 9
Checkpoint loaded at epoch: 0

MVSA-Multiple: densenet169
Fold: 0


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 24
Fold: 1
Checkpoint loaded at epoch: 0
Fold: 2
Checkpoint loaded at epoch: 0
Fold: 3
Checkpoint loaded at epoch: 0
Fold: 4
Checkpoint loaded at epoch: 0
Fold: 5
Checkpoint loaded at epoch: 0
Fold: 6
Checkpoint loaded at epoch: 0
Fold: 7
Checkpoint loaded at epoch: 0
Fold: 8
Checkpoint loaded at epoch: 0
Fold: 9
Checkpoint loaded at epoch: 0

MVSA-Multiple: densenet201
Fold: 0


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 24
Fold: 1
Checkpoint loaded at epoch: 0
Fold: 2
Checkpoint loaded at epoch: 0
Fold: 3
Checkpoint loaded at epoch: 0
Fold: 4
Checkpoint loaded at epoch: 0
Fold: 5
Checkpoint loaded at epoch: 0
Fold: 6
Checkpoint loaded at epoch: 0
Fold: 7
Checkpoint loaded at epoch: 0
Fold: 8
Checkpoint loaded at epoch: 0
Fold: 9
Checkpoint loaded at epoch: 0



,Loss,Accuracy,F1-macro,F1-weighted
resnet50,1.142633,0.101436,0.061396,0.184188
resnet101,1.203663,0.012043,0.007933,0.023799
densenet169,0.650834,0.748495,0.285386,0.856159
densenet201,0.549316,0.809634,0.298268,0.894804


## With merge MVSA data

In [ ]:
print('Both MVSA: With original image labels')
mvsa_histories3 = []
mvsa_scores3 = []
for i in range(len(feature_names)):
    print('Both MVSA:', feature_names[i])
    history, score = run_and_evaluate('merge-OL-' + feature_names[i], mvsa_features[i], mvsa_image_labels, verbose=0)
    mvsa_histories3.append(history)
    mvsa_scores3.append(score)
    print()
df_scores3 = pd.DataFrame(mvsa_scores3, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)

print('Both MVSA: With multimodal labels')
mvsa_histories4 = []
mvsa_scores4 = []
for i in range(len(feature_names)):
    print('Both MVSA:', feature_names[i])
    history, score = run_and_evaluate('merge-ML-' + feature_names[i], mvsa_features[i], mvsa_multimodal_labels, verbose=0)
    mvsa_histories4.append(history)
    mvsa_scores4.append(score)
    print()
df_scores4 = pd.DataFrame(mvsa_scores4, columns=['Loss', 'Accuracy', 'F1-macro', 'F1-weighted'], index=feature_names)

Both MVSA: With original image labels
Both MVSA: resnet50


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 5

Both MVSA: resnet101


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 5

Both MVSA: densenet169


...

Checkpoint loaded at epoch: 0

Both MVSA: densenet169


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 0

Both MVSA: densenet201


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Checkpoint loaded at epoch: 8



# Display results

In [ ]:
# print('With original image labels\n')
# display_dataframes((style_dataframe(df_single_scores), style_dataframe(df_multiple_scores), style_dataframe(df_average_scores)),
#                    names=['MVSA-Single', 'MVSA-Multiple', 'Average'])

In [ ]:
print('With multimodal labels\n')
display_dataframes((style_dataframe(df_single_scores2), style_dataframe(df_multiple_scores2), style_dataframe(df_average_scores2), style_dataframe(df_scores3), style_dataframe(df_scores4)),
                   names=['MVSA-Single', 'MVSA-Multiple', 'Average', 'All-image label','All-multimodal'])
print()

With multimodal labels



MVSA-Single,MVSA-Multiple,Average,All-image label,All-multimodal
,Loss,Accuracy,F1-macro,F1-weighted
resnet50,1.006928,0.567627,0.320978,0.466652
resnet101,1.030059,0.534368,0.296859,0.448299
densenet169,1.087860,0.509978,0.391734,0.528154
densenet201,1.003746,0.569845,0.411011,0.545499
,Loss,Accuracy,F1-macro,F1-weighted
resnet50,1.057283,0.478261,0.296883,0.451144
resnet101,1.024614,0.548766,0.314252,0.487315
densenet169,0.985290,0.548766,0.384404,0.527350
densenet201,0.995913,0.539365,0.364587,0.510505


In [ ]:
drop_columns = ['Loss', 'F1-macro']
def dataframe_to_display(df):
    return style_dataframe_out(np.round(df, 3).drop(columns=drop_columns))

In [ ]:
# if not os.path.exists('./tables'):
#     os.makedirs('./tables')
# open('./tables/single_image_scores.html', 'w').write(dataframe_to_display(df_single_scores2).to_html())
# open('./tables/multiple_image_scores.html', 'w').write(dataframe_to_display(df_multiple_scores2).to_html())
# open('./tables/average_image_scores.html', 'w').write(dataframe_to_display(df_average_scores2).to_html())

1457

In [ ]:
# print('With both MVSA merged together\n')
# display_dataframes((style_dataframe(df_scores3), style_dataframe(df_scores4)),
#                    names=['Original image labels', 'Multimodal labels'])

# Plots

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
WIDTH = 650
HEIGHT = 400
if not os.path.exists('./plots'):
    os.makedirs('./plots')

In [ ]:
df_single = pd.concat([df_single_scores2['Accuracy'], df_single_scores2['F1-weighted']], axis=1)
df_single['Đặc trưng'] = df_single.index
df_single['Bộ dữ liệu'] = 'MVSA-Single'
df_multiple = pd.concat([df_multiple_scores2['Accuracy'], df_multiple_scores2['F1-weighted']], axis=1)
df_multiple['Đặc trưng'] = df_multiple.index
df_multiple['Bộ dữ liệu'] = 'MVSA-Multiple'
df = pd.concat([df_single, df_multiple])

In [ ]:
import plotly.express as px

fig = px.bar(df, x="Đặc trưng", y=["Accuracy", "F1-weighted"], facet_col="Bộ dữ liệu", facet_col_spacing = 0.15)
fig.update_layout(barmode = 'group')
# fig.update_layout(bargap=0.2)
fig.update_layout(margin_b=25, margin_t=50, margin_l=25, margin_r=25, width=WIDTH, height=HEIGHT)
fig.update_layout(title='Biểu đồ kết quả đánh giá các đặc trưng của dữ liệu hình ảnh',
#               xaxis_title='Đặc trưng',
              yaxis_title='Điểm', legend_title='Độ đo')
#fig.write_html('./plots/image-scores-plot.html')
fig.show()

# Dratfs

In [ ]:
# # load separate
# mvsa_single_images, mvsa_multiple_images = load_mvsa_images()
# mvsa_single_xception, mvsa_multiple_xception = load_mvsa_feature('xception')
# mvsa_single_vgg16, mvsa_multiple_vgg16 = load_mvsa_feature('vgg16')
# mvsa_single_vgg19, mvsa_multiple_vgg19 = load_mvsa_feature('vgg19')
# mvsa_single_resnet50, mvsa_multiple_resnet50 = load_mvsa_feature('resnet50')
# mvsa_single_resnet101, mvsa_multiple_resnet101 = load_mvsa_feature('resnet101')
# mvsa_single_resnet152, mvsa_multiple_resnet152 = load_mvsa_feature('resnet152')
# mvsa_single_densenet121, mvsa_multiple_densenet121 = load_mvsa_feature('densenet121')
# mvsa_single_densenet169, mvsa_multiple_densenet169 = load_mvsa_feature('densenet169')
# mvsa_single_densenet201, mvsa_multiple_densenet201 = load_mvsa_feature('densenet201')

# # load merge
# mvsa_images = merge_mvsa(mvsa_single_images, mvsa_multiple_images)
# mvsa_xception = merge_mvsa(mvsa_single_xception, mvsa_multiple_xception)
# mvsa_vgg16 = merge_mvsa(mvsa_single_vgg16, mvsa_multiple_vgg16)
# mvsa_vgg19 = merge_mvsa(mvsa_single_vgg19, mvsa_multiple_vgg19)
# mvsa_resnet50 = merge_mvsa(mvsa_single_resnet50, mvsa_multiple_resnet50)
# mvsa_resnet101 = merge_mvsa(mvsa_single_resnet101, mvsa_multiple_resnet101)
# mvsa_resnet152 = merge_mvsa(mvsa_single_resnet152, mvsa_multiple_resnet152)
# mvsa_densenet121 = merge_mvsa(mvsa_single_densenet121, mvsa_multiple_densenet121)
# mvsa_densenet169 = merge_mvsa(mvsa_single_densenet169, mvsa_multiple_densenet169)
# mvsa_densenet201 = merge_mvsa(mvsa_single_densenet201, mvsa_multiple_densenet201)

# # prepare all features data
# feature_names = ['cnn', 'xception', 'vgg16', 'vgg19', 'resnet50', 'resnet101', 'resnet152', 'densenet121', 'densenet169', 'densenet201']

# mvsa_single_features = [mvsa_single_images,
#                         mvsa_single_xception,
#                         mvsa_single_vgg16, mvsa_single_vgg19,
#                         mvsa_single_resnet50, mvsa_single_resnet101, mvsa_single_resnet152,
#                         mvsa_single_densenet121, mvsa_single_densenet169, mvsa_single_densenet201]

# mvsa_multiple_features = [mvsa_multiple_images,
#                           mvsa_multiple_xception,
#                           mvsa_multiple_vgg16, mvsa_multiple_vgg19,
#                           mvsa_multiple_resnet50, mvsa_multiple_resnet101, mvsa_multiple_resnet152,
#                           mvsa_multiple_densenet121, mvsa_multiple_densenet169, mvsa_multiple_densenet201]

# mvsa_features = [mvsa_images,
#                  mvsa_xception,
#                  mvsa_vgg16, mvsa_vgg19,
#                  mvsa_resnet50, mvsa_resnet101, mvsa_resnet152,
#                  mvsa_densenet121, mvsa_densenet169, mvsa_densenet201]